## Implementing Bengio's: A Neural Probabilistic Language Model

To build a basic implementation, I structure the code around three main components:

1- The Embedding Layer (Matrix C): This is a lookup table of size [vocabulary_size, embedding_dimension]
For a given sequence, you look up the continuous vectors for the previous context words and concatenate them into a single long input vector


2- The Hidden Layer: You pass that concatenated vector into a standard linear layer (multiplying by weights and adding a bias) and apply a tanh activation function


3- The Output Layer: The hidden layer's output goes into a final linear layer, which expands the data back to the size of your entire vocabulary Finally, you apply a softmax function to turn those raw scores into probabilities that sum to 1


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

### Downloading and Accessing the Brown Corpus
The Brown Corpus was the first million-word electronic corpus of English, created in 1961 at Brown University.

In [ ]:
import nltk
nltk.download('brown')
from nltk.corpus import brown

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!


In [ ]:
# View the categories in the Brown corpus
print(f"Categories: {brown.categories()}\n")

# Sample some words from the 'news' category
news_text = brown.words(categories='news')
print(f"First 10 words in 'news': {news_text[:10]}")

Categories: ['adventure', 'belles_lettres', 'editorial', 'fiction', 'government', 'hobbies', 'humor', 'learned', 'lore', 'mystery', 'news', 'religion', 'reviews', 'romance', 'science_fiction']

First 10 words in 'news': ['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', 'Friday', 'an', 'investigation', 'of']


In [ ]:
len(news_text)

100554

In [ ]:
enumerate(news_text)

In [ ]:
vocab = list(news_text.iterate_from(0))
vicab_size = len(vocab)

In [ ]:
char2num = {word: i for i, word in enumerate(vocab)}
num2char = {i: word for word, i in char2num.items()}

In [ ]:
from nltk import FreqDist

# news_text is already defined as brown.words(categories='news')
fdist = FreqDist(news_text)

# Display the 20 most common words
print("20 Most Common Words in 'news' category:")
for word, frequency in fdist.most_common(20):
    print(f"{word}: {frequency}")

20 Most Common Words in 'news' category:
the: 5580
,: 5188
.: 4030
of: 2849
and: 2146
to: 2116
a: 1993
in: 1893
for: 943
The: 806
that: 802
``: 732
is: 732
was: 717
'': 702
on: 657
at: 598
with: 545
be: 526
by: 497


In [ ]:
# # build the training set
# from collections import deque

# block_size = 3    # context length
# X, y = [], []

# # We iterate through the sequence of words in the vocab
# # context is a sliding window of previous word indices
# context = deque([0] * block_size, maxlen=block_size)

# for word in vocab:
#     ix = char2num[word]
#     X.append(list(context))
#     y.append(ix)

#     context_words = [num2char[i] if i != 0 else "<PAD>" for i in context]
#     # print(f"{' '.join(context_words)} -----> {word}")
#     context.append(ix)

# X = torch.tensor(X)
# y = torch.tensor(y)

In [ ]:
# split the data
train_split = int(0.8 * len(vocab))
valid_split = int(0.9 * len(vocab))

In [ ]:
import random
from collections import deque

random.shuffle(vocab)


########################

def build_dataset(vocab):
  # build the training set

  block_size = 3    # context length
  X, y = [], []

  # We iterate through the sequence of words in the vocab
  # context is a sliding window of previous word indices
  context = deque([0] * block_size, maxlen=block_size)

  for word in vocab:
      ix = char2num[word]
      X.append(list(context))
      y.append(ix)

      context_words = [num2char[i] if i != 0 else "<PAD>" for i in context]
      # print(f"{' '.join(context_words)} -----> {word}")
      context.append(ix)

  X = torch.tensor(X)
  y = torch.tensor(y)

  return X, y

In [ ]:
X_train, y_train = build_dataset(vocab[:train_split])
X_valid, y_valid = build_dataset(vocab[train_split:valid_split])
X_test,  y_test  = build_dataset(vocab[valid_split:])

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Create Datasets
train_ds = TensorDataset(X_train, y_train)
valid_ds = TensorDataset(X_valid, y_valid)

# Define batch size
batch_size = 256

# Create DataLoaders
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False)

# Quick check
xb, yb = next(iter(train_loader))
print(f"Batch X shape: {xb.shape}")
print(f"Batch y shape: {yb.shape}")

Batch X shape: torch.Size([256, 3])
Batch y shape: torch.Size([256])


In [ ]:
class BengioNeuralLM(nn.Module):

    def __init__(self, vocab_size, embedding_dim, context_size, hidden_size, use_direct_connection=False):
        super().__init__()
        self.context_size = context_size
        self.use_direct_connection = use_direct_connection

        # Matrix C in the paper
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # Matrix H and bias d in the paper
        self.fc1 = nn.Linear(context_size * embedding_dim, hidden_size)

        # Matrix U and bias b in the paper
        self.fc2 = nn.Linear(hidden_size, vocab_size)

        # Optional Matrix W in the paper
        if self.use_direct_connection:
            self.direct = nn.Linear(context_size * embedding_dim, vocab_size, bias=False)

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        # Kaiming initialization for layers followed by Tanh
        nn.init.kaiming_normal_(self.fc1.weight, mode='fan_in', nonlinearity='tanh')
        nn.init.constant_(self.fc1.bias, 0)

        # Initialize output layer weights to be small to avoid high initial loss
        nn.init.xavier_normal_(self.fc2.weight)
        nn.init.constant_(self.fc2.bias, 0)

        # Initialize embeddings with a small standard deviation
        nn.init.normal_(self.embedding.weight, mean=0, std=0.01)

        if self.use_direct_connection:
            nn.init.xavier_normal_(self.direct.weight)

    def forward(self, x):
        # Input x shape: (batch_size, context_size)

        # 1. Look up embeddings
        # Shape: (batch_size, context_size, embedding_dim)
        embeds = self.embedding(x)

        # 2. Concatenate the context word vectors
        # Shape: (batch_size, context_size * embedding_dim)
        embeds_flat = embeds.view(embeds.size(0), -1)

        # 3. Hidden layer with tanh activation
        # Shape: (batch_size, hidden_size)
        h = torch.tanh(self.fc1(embeds_flat))

        # 4. Output layer (Logits)
        # Shape: (batch_size, vocab_size)
        logits = self.fc2(h)

        # 5. Add direct connection if enabled
        if self.use_direct_connection:
            logits += self.direct(embeds_flat)

        return logits

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Hyperparameters
CONTEXT_SIZE = 3
EMBEDDING_DIM = 10
HIDDEN_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 50

# Initialize model, criterion, and optimizer
model = BengioNeuralLM(vocab_size=vicab_size,
                       embedding_dim=EMBEDDING_DIM,
                       context_size=CONTEXT_SIZE,
                       hidden_size=HIDDEN_SIZE,
                       use_direct_connection=True).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

Using device: cuda


In [ ]:
sum(p.numel() for p in model.parameters())

7341434

In [ ]:
# Training Loop
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch_idx, (data, target) in enumerate(train_loader):
        # Move data to device
        data, target = data.to(device), target.to(device)

        # Forward pass
        logits = model(data)
        loss = criterion(logits, target)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}")

Epoch [1/50], Loss: 7.8179
Epoch [2/50], Loss: 6.9929
Epoch [3/50], Loss: 6.2659
Epoch [4/50], Loss: 5.1975
Epoch [5/50], Loss: 4.1937
Epoch [6/50], Loss: 3.4784
Epoch [7/50], Loss: 3.0145
Epoch [8/50], Loss: 2.6926
Epoch [9/50], Loss: 2.4536
Epoch [10/50], Loss: 2.2702
Epoch [11/50], Loss: 2.1193
Epoch [12/50], Loss: 1.9922
Epoch [13/50], Loss: 1.8846
Epoch [14/50], Loss: 1.7920
Epoch [15/50], Loss: 1.7073
Epoch [16/50], Loss: 1.6295
Epoch [17/50], Loss: 1.5603
Epoch [18/50], Loss: 1.4941
Epoch [19/50], Loss: 1.4322
Epoch [20/50], Loss: 1.3710
Epoch [21/50], Loss: 1.3151
Epoch [22/50], Loss: 1.2569
Epoch [23/50], Loss: 1.2021
Epoch [24/50], Loss: 1.1480
Epoch [25/50], Loss: 1.0937
Epoch [26/50], Loss: 1.0412
Epoch [27/50], Loss: 0.9899
Epoch [28/50], Loss: 0.9392
Epoch [29/50], Loss: 0.8896
Epoch [30/50], Loss: 0.8419
Epoch [31/50], Loss: 0.7955
Epoch [32/50], Loss: 0.7484
Epoch [33/50], Loss: 0.7093
Epoch [34/50], Loss: 0.6672
Epoch [35/50], Loss: 0.6299
Epoch [36/50], Loss: 0.5925
E

### Model Evaluation
Now we evaluate the model on the test set that was held out during training and validation.

In [ ]:
def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            logits = model(data)
            loss = criterion(logits, target)
            total_loss += loss.item()

            # Calculate accuracy
            probs = F.softmax(logits, dim=1)
            predictions = torch.argmax(probs, dim=1)
            correct += (predictions == target).sum().item()
            total += target.size(0)

    avg_loss = total_loss / len(data_loader)
    accuracy = correct / total
    return avg_loss, accuracy



# Run evaluation
val_loss, val_acc = evaluate(model, valid_loader, criterion, device)

print(f"val Loss: {val_loss:.4f}")
print(f"val Accuracy: {val_acc:.2%}")

val Loss: 23.7318
val Accuracy: 1.75%


In [ ]:
test_ds = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

# Run evaluation
test_loss, test_acc = evaluate(model, test_loader, criterion, device)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2%}")

Test Loss: 23.7476
Test Accuracy: 1.81%


In [ ]:
# Early Stopping Configuration
patience = 5  # Number of epochs to wait without improvement
best_val_loss = float('inf')
epochs_no_improve = 0

# Training Loop
for epoch in range(EPOCHS):
    # ========================
    # 1. Training Phase
    # ========================
    model.train()
    total_train_loss = 0

    for batch_idx, (data, target) in enumerate(train_loader):
        # Move data to device
        data, target = data.to(device), target.to(device)

        # Forward pass
        optimizer.zero_grad() # Moved up (best practice to zero before forward pass)
        logits = model(data)
        loss = criterion(logits, target)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # ========================
    # 2. Validation Phase
    # ========================
    model.eval() # Set model to evaluation mode (affects Dropout/BatchNorm)
    total_val_loss = 0

    # Disable gradient computation for validation to save memory and compute
    with torch.no_grad():
        for data, target in valid_loader: # Assuming you have a val_loader defined
            data, target = data.to(device), target.to(device)

            logits = model(data)
            loss = criterion(logits, target)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(valid_loader)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # ========================
    # 3. Early Stopping Logic
    # ========================
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0

        # Optional but recommended: Save the best model state
        # torch.save(model.state_dict(), 'best_model.pt')
    else:
        epochs_no_improve += 1
        print(f"Validation loss did not improve for {epochs_no_improve} epoch(s).")

        if epochs_no_improve >= patience:
            print(f"Early stopping triggered! Halting training at epoch {epoch+1}.")
            break # Exit the epoch loop

Epoch [1/50] | Train Loss: 8.8019 | Val Loss: 7.6141
Epoch [2/50] | Train Loss: 7.2205 | Val Loss: 7.5375
Epoch [3/50] | Train Loss: 7.0928 | Val Loss: 7.5489
Validation loss did not improve for 1 epoch(s).
Epoch [4/50] | Train Loss: 7.0398 | Val Loss: 7.5818
Validation loss did not improve for 2 epoch(s).
Epoch [5/50] | Train Loss: 6.9968 | Val Loss: 7.6243
Validation loss did not improve for 3 epoch(s).
Epoch [6/50] | Train Loss: 6.9541 | Val Loss: 7.6740
Validation loss did not improve for 4 epoch(s).
Epoch [7/50] | Train Loss: 6.9036 | Val Loss: 7.7385
Validation loss did not improve for 5 epoch(s).
Early stopping triggered! Halting training at epoch 7.


In [ ]:
class BengioNeuralLM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, context_size, hidden_size, use_direct_connection=False):
        super().__init__()
        self.context_size = context_size
        self.use_direct_connection = use_direct_connection
        
        # Matrix C in the paper
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Matrix H and bias d in the paper
        # Note the input dimension is context_size * embedding_dim
        self.fc1 = nn.Linear(context_size * embedding_dim, hidden_size)
        
        # Matrix U and bias b in the paper
        self.fc2 = nn.Linear(hidden_size, vocab_size)
        
        # Optional Matrix W in the paper
        if self.use_direct_connection:
            self.direct = nn.Linear(context_size * embedding_dim, vocab_size, bias=False)

    def forward(self, x):
        # Input x shape: (batch_size, context_size)
        
        # 1. Look up embeddings
        # Shape: (batch_size, context_size, embedding_dim)
        embeds = self.embedding(x)
        
        # 2. Concatenate the context word vectors
        # Shape: (batch_size, context_size * embedding_dim)
        # Using .view() or .reshape() flattens the sequence of embeddings into a single vector per batch
        embeds_flat = embeds.view(embeds.size(0), -1)
        
        # 3. Hidden layer with tanh activation
        # Shape: (batch_size, hidden_size)
        h = F.tanh(self.fc1(embeds_flat))
        
        # 4. Output layer (Logits)
        # Shape: (batch_size, vocab_size)
        logits = self.fc2(h)
        
        # 5. Add direct connection if enabled
        if self.use_direct_connection:
            logits += self.direct(embeds_flat)
            
        # Return raw logits to be used with nn.CrossEntropyLoss
        return logits

In [ ]:
class NeuralLM(nn.Module):
    
    def __init__(self, embedding_dim, hidden_size, vocab_size, context_size, direct_connection=False):
        super().__init__()
        self.context_size = context_size
        self.hidden_size = hidden_size
        self.direct_connection = direct_connection
        
        #layers
        # matrix C - lookup table (vocab size, embedding_dim)
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # 1st hidden layer
        self.fc1 = nn.Linear(context_size * embedding_dim, hidden_size)
        
        # output layer
        self.fc2 = nn.Linear(hidden_size, vocab_size)
        
        if self.direct_connection:
            self.direct_layer = nn.Linear(context_size * embedding_dim, vocab_size)
            
    def forward(self, x):
        # transform raw int idices to dense vectors
        embeds = self.embedding(x) # embed shape (1, 3, 10) 1 batch, it has 3 word indices, each has a 10 dim vector, so each batch has a 30 dim vector: context * embed_dim = 3 * 10
        
        # concat the 3 different embedding vector into 1 vector
        concat_embed = embeds.view(embeds.shape[0], -1)  # collapses the last 2 dims into 1 dim:
        
        z = self.fc1(concat_embed)
        h = F.tanh(z)
        logits = self.fc2(h)
        
        if self.direct_connection:
            logits += self.direct_layer(concat_embed)
        
        return logits
        